In [0]:
%pip uninstall -y numpy pandas pyarrow torch transformers tokenizers huggingface-hub safetensors fsspec sympy networkx opencv-python opencv-python-headless typing-extensions

%pip install -i https://mirrors.tuna.tsinghua.edu.cn/pypi/web/simple \
  --extra-index-url https://download.pytorch.org/whl/cpu \
  "numpy==1.26.4" \
  "pandas==2.2.2" \
  "pyarrow==16.1.0" \
  "transformers==4.53.0" \
  "opencv-python==4.10.0.84" \
  "typing-extensions==4.15.0" \
  "sympy==1.14.0" \
  "networkx==3.4.2" \
  "fsspec==2025.3.0" \
  "huggingface-hub==0.33.1" \
  "tokenizers==0.21.2" \
  "safetensors==0.5.3" \
  "torch==2.7.1+cpu"

dbutils.library.restartPython()

import numpy
import pandas
import cv2
import torch
import transformers

print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("cv2:", cv2.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
print("transformers:", transformers.__version__)

from transformers import ChineseCLIPProcessor, ChineseCLIPModel
print("ChineseCLIP import OK")

In [0]:
from transformers import ChineseCLIPProcessor, ChineseCLIPModel
from PIL import Image
import requests
import torch
import cv2
import re
from io import BytesIO
import pandas as pd
import os
import json
from pyspark.sql.types import ArrayType, DoubleType, StructField, StructType

In [0]:
def embed_text(processor, model, text_list, normalize=True, to_numpy=False):
    inputs = processor(text=text_list, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        emb = model.get_text_features(**inputs)
        if normalize:
          emb = emb / emb.norm(p=2, dim=-1, keepdim=True)
    return emb.cpu().numpy() if to_numpy else emb.cpu()

def embed_image(processor, model, images, normalize=True, to_numpy=False):
    inputs = processor(images=images, return_tensors="pt", padding=True).to(device)
    with torch.no_grad():
        emb = model.get_image_features(**inputs)
        if normalize:
          emb = emb / emb.norm(p=2, dim=-1, keepdim=True)
    return emb.cpu().numpy() if to_numpy else emb.cpu()

def get_images(image_paths_or_urls):
    images = []
    for path in image_paths_or_urls:
        try:
            if path.startswith("http"):
                response = requests.get(path, timeout=30)
                response.raise_for_status()
                img = Image.open(BytesIO(response.content)).convert("RGB")
            else:
                img = Image.open(path).convert("RGB")
            images.append(img)
        except Exception as exc:
            print(f"Fail to open the image file: {path}. Error: {exc}")
            continue
    return images

def extract_first_frame(video_path):
    """Extracts the first frame of a video as a PIL.Image"""
    if not video_path or not isinstance(video_path, str):
        return None

    cap = cv2.VideoCapture(video_path)
    success, frame = cap.read()
    cap.release()
    if not success or frame is None:
        print(f"Could not read first frame from {video_path}")
        return None
    else:
        # Convert from BGR (OpenCV default) to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        return Image.fromarray(frame_rgb)

In [0]:
def embed_post_content(processor: ChineseCLIPProcessor, model: ChineseCLIPModel, text_series: pd.Series, pics_series: pd.Series, vids_series: pd.Series) -> list:
    outputs = []

    for text, pics, vids in zip(text_series, pics_series, vids_series):
        embeddings = []

        # text embedding
        if text is not None:
            txt_feat = embed_text(processor, model, text_list=[text], normalize=True)
            embeddings.append(txt_feat[0])

        # image embedding
        if pics is not None:
            pic_urls = pics.tolist()
            for pic_url in pic_urls:
                images = get_images(image_paths_or_urls=[pic_url]) 
                if len(images) > 0:
                    img_feats = embed_image(processor, model, images=images, normalize=True)
                    embeddings.append(img_feats[0])

        # video (first frame) embedding
        if vids is not None:
            vid_urls = vids.tolist()
            for vid_url in vid_urls:
                frame_image = extract_first_frame(vid_url)
                if frame_image:
                    vid_feats = embed_image(processor, model, images=[frame_image], normalize=True)
                    embeddings.append(vid_feats[0])

        # pooling: average
        if embeddings:
            emb_stack = torch.stack(embeddings)
            pooled = emb_stack.mean(dim=0)  # or emb_stack.sum(dim=0)
            outputs.append(pooled.cpu().tolist())
        else:
            outputs.append([0.0] * model.config.projection_dim)

    return outputs

In [0]:
def is_html(text):
    """Check if text contains HTML tags."""
    if pd.isna(text) or not isinstance(text, str):
        return False
    
    html_pattern = re.compile(
        r'<([A-Za-z][A-Za-z0-9]*)\b[^>]*>(.*?)</\1>|<[A-Za-z][A-Za-z0-9]*\b[^>]*/>|<!DOCTYPE\s+html|<!--.*?-->',
        re.DOTALL | re.IGNORECASE
    )
    return bool(html_pattern.search(text))

def truncate_text(text, threshold):
    if text:
        return text[:threshold] if len(text) > threshold else text
    else:
        return None

In [0]:
def embed_post_content_table(spark, processor, model, run_date, db, table_names, trunc_len=512):
    post_content_sdf = spark.sql("""
                    select * 
                    from {db}.{post_content} 
                    where partition_date = '{run_date}' 
                    """.format(run_date = run_date,
                              db = db,
                              post_content = table_names["post_content"]))
    df = post_content_sdf.toPandas()

    if df.empty:
        return spark.createDataFrame([], StructType([
            StructField("post_id", post_content_sdf.schema["post_id"].dataType, True),
            StructField("pooled_embedding", ArrayType(DoubleType(), True), True),
        ]))
    
    # Replace HTML content in a pandas DataFrame column with null values.
    df['text_content'] = df['text_content'].apply(lambda x: None if is_html(x) else x)

    # Truncate long text to meet the requirement of embedding model input size
    df['text_content'] = df['text_content'].apply(lambda x: truncate_text(x, trunc_len))

    df["pooled_embedding"] = embed_post_content(processor, model, df['text_content'], df['pic_url'], df['video_url'])

    return spark.createDataFrame(df[["post_id","pooled_embedding"]])

In [0]:
def write_post_embedding_table(spark, processor, model, run_date, db, table_names, trunc_len=512):
    sdf_emb = embed_post_content_table(spark, processor, model, run_date, db, table_names, trunc_len)
    sdf_emb.createOrReplaceTempView("post_embedding")
    sdf = spark.sql("""
                    INSERT OVERWRITE TABLE {db}.{post_embedding} PARTITION (partition_date = '{run_date}')
                    select * from post_embedding
                    """.format(run_date = run_date,
                               db = db,
                               post_embedding = table_names["post_embedding"]))
    

In [0]:
# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter run_date:{}".format(run_date))

config_path = dbutils.widgets.get("config_path")
with open(config_path, "r") as file:
    config = json.load(file)
db, table_names = config['DATABASE'], config['TABLE_NAMES']

In [0]:
# Load model
model_path = config["CLIP_MODEL_PATH"]
processor = ChineseCLIPProcessor.from_pretrained(model_path)
model = ChineseCLIPModel.from_pretrained(model_path)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [0]:
write_post_embedding_table(spark, processor, model, run_date, db, table_names)